Artigo principal: https://dl.acm.org/doi/10.1145/3698300.3698316#Bib0015 

Dataset das instancias (citado pelo paper): https://github.com/wrqccc/FJSP-DRL/tree/main/data/BenchData/Brandimarte

![alt text](./imgs/paper_table2.png)

## Leitura da Instância

### Formato do arquivo `.fjs` (Flexible Job Shop)

- **Linha 1**: `num_jobs  num_machines  avg_machines_per_op`
- **Linhas seguintes** (uma por job):
  - Primeiro número: `n_ops` (quantidade de operações do job)
  - Para cada operação: `n_alt` seguido de `n_alt` pares `(máquina  tempo_processamento)`

As máquinas são indexadas a partir de **1** no arquivo.

In [6]:
from dataclasses import dataclass, field
from typing import List, Dict, Tuple


@dataclass
class Operation:
    """Uma operação de um job, com múltiplas alternativas de máquina."""
    job_id: int
    op_index: int
    # Dict: machine_id (1-based) -> processing_time
    machines: Dict[int, int] = field(default_factory=dict)

    def __repr__(self):
        opts = ", ".join(f"M{m}:{t}" for m, t in sorted(self.machines.items()))
        return f"Op({self.job_id},{self.op_index})[{opts}]"


@dataclass
class Job:
    """Um job composto por uma sequência de operações."""
    job_id: int
    operations: List[Operation] = field(default_factory=list)

    def __repr__(self):
        return f"Job {self.job_id}: {self.operations}"


@dataclass
class FJSPInstance:
    """Instância completa do Flexible Job Shop Scheduling Problem."""
    num_jobs: int
    num_machines: int
    jobs: List[Job] = field(default_factory=list)


def parse_fjs(filepath: str) -> FJSPInstance:
    """Lê um arquivo .fjs e retorna uma FJSPInstance."""
    with open(filepath, 'r') as f:
        tokens = f.read().split()

    idx = 0

    # Cabeçalho
    num_jobs = int(tokens[idx]);     idx += 1
    num_machines = int(tokens[idx]); idx += 1
    _avg = int(tokens[idx]);         idx += 1  # avg machines per op (ignorado na leitura)

    instance = FJSPInstance(num_jobs=num_jobs, num_machines=num_machines)

    for j in range(num_jobs):
        n_ops = int(tokens[idx]); idx += 1
        job = Job(job_id=j + 1)

        for o in range(n_ops):
            n_alt = int(tokens[idx]); idx += 1
            op = Operation(job_id=j + 1, op_index=o + 1)

            for _ in range(n_alt):
                machine = int(tokens[idx]); idx += 1
                time    = int(tokens[idx]); idx += 1
                op.machines[machine] = time

            job.operations.append(op)

        instance.jobs.append(job)

    return instance


# Lê a primeira instância
instance = parse_fjs('./instancias/BrandimarteMk1.fjs')
print(f"Instância BrandimarteMk1")
print(f"  Jobs     : {instance.num_jobs}")
print(f"  Máquinas : {instance.num_machines}")
print()

Instância BrandimarteMk1
  Jobs     : 10
  Máquinas : 6



In [7]:
# Exibe os jobs e operações da instância
for job in instance.jobs:
    print(f"Job {job.job_id:>2}  ({len(job.operations)} operações):")
    for op in job.operations:
        opts = "  |  ".join(f"M{m} → t={t}" for m, t in sorted(op.machines.items()))
        print(f"    Op {op.op_index}: [{opts}]")
    print()

Job  1  (6 operações):
    Op 1: [M1 → t=5  |  M3 → t=4]
    Op 2: [M2 → t=1  |  M3 → t=5  |  M5 → t=3]
    Op 3: [M3 → t=4  |  M6 → t=2]
    Op 4: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 5: [M3 → t=1]
    Op 6: [M3 → t=6  |  M4 → t=3  |  M6 → t=6]

Job  2  (5 operações):
    Op 1: [M2 → t=6]
    Op 2: [M3 → t=1]
    Op 3: [M1 → t=2]
    Op 4: [M2 → t=6  |  M4 → t=6]
    Op 5: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]

Job  3  (5 operações):
    Op 1: [M2 → t=6]
    Op 2: [M3 → t=4  |  M6 → t=2]
    Op 3: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 4: [M2 → t=6  |  M3 → t=4  |  M6 → t=6]
    Op 5: [M1 → t=1  |  M5 → t=5]

Job  4  (5 operações):
    Op 1: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 2: [M2 → t=6]
    Op 3: [M3 → t=1]
    Op 4: [M2 → t=1  |  M3 → t=5  |  M5 → t=3]
    Op 5: [M3 → t=4  |  M6 → t=2]

Job  5  (6 operações):
    Op 1: [M2 → t=1  |  M3 → t=5  |  M5 → t=3]
    Op 2: [M1 → t=1  |  M2 → t=6  |  M6 → t=5]
    Op 3: [M2 → t=6]
    Op 4: [M1 → t=5  |  M3 → t=4]
    O